In [ ]:
import pandas as pd
import os

#####################
#  1) LOAD CSV FILES
#####################

# Example paths for the four validation CSVs
val_csv1 = "/path/to/workspace/Complete_set/Experiments/Results/FMmodel192/val/latest_FM_192.csv"
val_csv2 = "/path/to/workspace/Complete_set/Experiments/Results/patches_classificationmodel/val/best_centrecrop-rectumpatches.csv"
val_csv3 = "/path/to/workspace/Complete_set/Experiments/Results/FM224_weightchange3/val/best_FM_weightchange3.csv"
val_csv4 = "/path/to/workspace/Complete_set/Experiments/Results/FM224_haromonizedaxial_oldimage/val/best_FM_harmonize_axial.csv"

# Example paths for the four test CSVs
test_csv1 = "/path/to/workspace/Complete_set/Experiments/Results/FMmodel192/test/latest_FM_192.csv"
test_csv2 = "/path/to/workspace/Complete_set/Experiments/Results/patches_classificationmodel/test/best_centrecrop-rectumpatches.csv"
test_csv3 = "/path/to/workspace/Complete_set/Experiments/Results/FM224_weightchange3/test/best_FM_weightchange3.csv"
test_csv4 = "/path/to/workspace/Complete_set/Experiments/Results/FM224_haromonizedaxial_oldimage/test/best_FM_harmonize_axial.csv"

# Read them into separate DataFrames
df1_val = pd.read_csv(val_csv1)
df2_val = pd.read_csv(val_csv2)
df3_val = pd.read_csv(val_csv3)
df4_val = pd.read_csv(val_csv4)

df1_test = pd.read_csv(test_csv1)
df2_test = pd.read_csv(test_csv2)
df3_test = pd.read_csv(test_csv3)
df4_test = pd.read_csv(test_csv4)

###########################
# 2) RENAME COLUMNS TO AVOID CONFLICTS
###########################
# Each CSV has columns:
#   image_name, predicted_probability_evi, predicted_probability_mfi,
#   groud_truth_evi, groud_truth_mfi
df1_val["image_name"] = df1_val["image_name"].apply(lambda x: os.path.basename(str(x)))
df2_val["image_name"] = df2_val["image_name"].apply(lambda x: os.path.basename(str(x)))
df3_val["image_name"] = df3_val["image_name"].apply(lambda x: os.path.basename(str(x)))
df4_val["image_name"] = df4_val["image_name"].apply(lambda x: os.path.basename(str(x)))

df1_test["image_name"] = df1_test["image_name"].apply(lambda x: os.path.basename(str(x)))
df2_test["image_name"] = df2_test["image_name"].apply(lambda x: os.path.basename(str(x)))
df3_test["image_name"] = df3_test["image_name"].apply(lambda x: os.path.basename(str(x)))
df4_test["image_name"] = df4_test["image_name"].apply(lambda x: os.path.basename(str(x)))


# For the validation set:
df1_val = df1_val.rename(columns={
    'predicted_probability_evi': 'pred_evi_m1',
    'predicted_probability_mfi': 'pred_mfi_m1',
})

df2_val = df2_val.rename(columns={
    'predicted_probability_evi': 'pred_evi_m2',
    'predicted_probability_mfi': 'pred_mfi_m2',
})

df3_val = df3_val.rename(columns={
    'predicted_probability_evi': 'pred_evi_m3',
    'predicted_probability_mfi': 'pred_mfi_m3',
})

df4_val = df4_val.rename(columns={
    'predicted_probability_evi': 'pred_evi_m4',
    'predicted_probability_mfi': 'pred_mfi_m4',
})

# For the test set:
df1_test = df1_test.rename(columns={
    'predicted_probability_evi': 'pred_evi_m1',
    'predicted_probability_mfi': 'pred_mfi_m1',
})

df2_test = df2_test.rename(columns={
    'predicted_probability_evi': 'pred_evi_m2',
    'predicted_probability_mfi': 'pred_mfi_m2',
})

df3_test = df3_test.rename(columns={
    'predicted_probability_evi': 'pred_evi_m3',
    'predicted_probability_mfi': 'pred_mfi_m3',
})

df4_test = df4_test.rename(columns={
    'predicted_probability_evi': 'pred_evi_m4',
    'predicted_probability_mfi': 'pred_mfi_m4',
})

###############################
# 3) MERGE ALL MODEL PREDICTIONS
###############################
# Merge on 'image_name'; keep ground-truth from the first DataFrame
# (assuming they are the same in all CSV files).

# Validation merge
merged_val = df1_val.merge(
    df2_val[['image_name','pred_evi_m2','pred_mfi_m2']], on='image_name'
)
merged_val = merged_val.merge(
    df3_val[['image_name','pred_evi_m3','pred_mfi_m3']], on='image_name'
)
merged_val = merged_val.merge(
    df4_val[['image_name','pred_evi_m4','pred_mfi_m4']], on='image_name'
)

# Test merge
merged_test = df1_test.merge(
    df2_test[['image_name','pred_evi_m2','pred_mfi_m2']], on='image_name'
)
merged_test = merged_test.merge(
    df3_test[['image_name','pred_evi_m3','pred_mfi_m3']], on='image_name'
)
merged_test = merged_test.merge(
    df4_test[['image_name','pred_evi_m4','pred_mfi_m4']], on='image_name'
)

############################################
# 4) CREATE SIMPLE AVERAGE ENSEMBLE COLUMNS
############################################
import numpy as np

# For validation set
# merged_val['ensemble_evi'] = np.mean(
#     merged_val[['pred_evi_m1','pred_evi_m3','pred_evi_m4']], axis=1
# ) #evi discurd the patches_classificationmodel

# merged_val['ensemble_mfi'] = np.mean(
#     merged_val[['pred_mfi_m1','pred_mfi_m2','pred_mfi_m3','pred_mfi_m4']], axis=1
# )


# Example weights for EVI
weights_evi = [0.2, 0.0, 0.2, 0.6]

# Extract the model columns as a numpy array
pred_evis_val = merged_val[['pred_evi_m1','pred_evi_m2','pred_evi_m3','pred_evi_m4']].values

# Use np.average along axis=1, passing your custom weights
merged_val['ensemble_evi'] = np.average(pred_evis_val, axis=1, weights=weights_evi)

# Similarly for MFI, maybe you want a different weighting:
weights_mfi = [0.05, 0.85 , 0.05, 0.05]
pred_mfis_val = merged_val[['pred_mfi_m1','pred_mfi_m2','pred_mfi_m3','pred_mfi_m4']].values
merged_val['ensemble_mfi'] = np.average(pred_mfis_val, axis=1, weights=weights_mfi)



# Extract the model columns as a numpy array
pred_evis_val = merged_test[['pred_evi_m1','pred_evi_m2','pred_evi_m3','pred_evi_m4']].values

# Use np.average along axis=1, passing your custom weights
merged_test['ensemble_evi'] = np.average(pred_evis_val, axis=1, weights=weights_evi)


pred_mfis_val = merged_test[['pred_mfi_m1','pred_mfi_m2','pred_mfi_m3','pred_mfi_m4']].values
merged_test['ensemble_mfi'] = np.average(pred_mfis_val, axis=1, weights=weights_mfi)


## For test set
#merged_test['ensemble_evi'] = np.mean(
#    merged_test[['pred_evi_m1','pred_evi_m3','pred_evi_m4']], axis=1
#)
#merged_test['ensemble_mfi'] = np.mean(
#    merged_test[['pred_mfi_m1','pred_mfi_m2','pred_mfi_m3','pred_mfi_m4']], axis=1
#)

# Now merged_val and merged_test each contain:
#   image_name, pred_evi_m1..m4, pred_mfi_m1..m4,
#   groud_truth_evi, groud_truth_mfi,
#   ensemble_evi, ensemble_mfi

########################
# 5) OPTIONAL: SAVE ENSEMBLE CSV
########################
merged_val.to_csv("/path/to/workspace/Complete_set/Experiments/Results/val_ensemble.csv", index=False)
merged_test.to_csv("/path/to/workspace/Complete_set/Experiments/Results/test_ensemble.csv", index=False)
print('done')


In [ ]:
import numpy as np
import sklearn.metrics as skm

def calculate_metrics(y_true, y_pred, threshold=0.5):
    """
    y_true: 1D array-like, ground truth (0 or 1).
    y_pred: 1D array-like of probabilities or continuous scores.
    threshold: Classification cutoff for converting y_pred to binary predictions.

    Returns a dict of metrics including:
      - AUC
      - Balanced Accuracy
      - Sensitivity (Recall for positive class=1)
      - Specificity (Recall for negative class=0)
      - F1
      - A custom 'fScore' combining them (example weighting)
    """
    # Convert to numpy arrays
    y_true = np.array(y_true, dtype=np.int32)
    y_pred = np.array(y_pred, dtype=np.float32)

    # Binary predictions based on threshold
    y_pred_binary = (y_pred > threshold).astype(int)

    # AUC needs probability scores or continuous predictions
    auc = skm.roc_auc_score(y_true, y_pred)

    # Balanced accuracy is average of recall for each class
    balanced_acc = skm.balanced_accuracy_score(y_true, y_pred_binary)

    # Sensitivity = recall for positive label=1
    sensitivity = skm.recall_score(y_true, y_pred_binary, pos_label=1)

    # Specificity = recall for negative label=0
    specificity = skm.recall_score(y_true, y_pred_binary, pos_label=0)

    # F1 score with predicted binary
    f1 = skm.f1_score(y_true, y_pred_binary)

    # Example custom formula
    fScore = 0.4 * auc + 0.2 * balanced_acc + 0.2 * sensitivity + 0.2 * specificity

    return {
        'auc': auc,
        'balanced_acc': balanced_acc,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'f1': f1,
        'fScore': fScore
    }


# ------------------------
# Example usage
# ------------------------
if __name__ == "__main__":
    # 1) Read your ensemble CSV (created previously)
    df_ensemble = pd.read_csv("/path/to/workspace/Complete_set/Experiments/Results/val_ensemble.csv")
    # Suppose it has columns:
    #   image_name, ensemble_evi, ensemble_mfi,
    #   ground_truth_evi, ground_truth_mfi, ...

    # 2) Compute metrics for EVI
    evi_metrics = calculate_metrics(
        y_true=df_ensemble["ground_truth_evi"], 
        y_pred=df_ensemble["ensemble_evi"], 
        threshold=0.5
    )

    # 3) Compute metrics for MFI
    mfi_metrics = calculate_metrics(
        y_true=df_ensemble["ground_truth_mfi"], 
        y_pred=df_ensemble["ensemble_mfi"], 
        threshold=0.5
    )

    # 4) Print or store the results
    print("=== EVI Metrics ===")
    for k, v in evi_metrics.items():
        print(f"{k}: {v:.4f}")

    print("\n=== MFI Metrics ===")
    for k, v in mfi_metrics.items():
        print(f"{k}: {v:.4f}")


In [ ]:
  ##############################
#  2) Load Test Ensemble CSV
    ##############################
df_test = pd.read_csv("/path/to/workspace/Complete_set/Experiments/Results/test_ensemble.csv")
# Make sure it has the same columns

print("\n=== TEST METRICS ===")
# EVI metrics
test_evi_metrics = calculate_metrics(
        y_true=df_test["ground_truth_evi"],
        y_pred=df_test["ensemble_evi"],
        threshold=0.5
    )
# MFI metrics
test_mfi_metrics = calculate_metrics(
        y_true=df_test["ground_truth_mfi"],
        y_pred=df_test["ensemble_mfi"],
        threshold=0.5
    )

print("EVI metrics:", test_evi_metrics)
print("MFI metrics:", test_mfi_metrics)